# Stage 1: Logistic Regression

This first stage treats the classifier as one sigmoid neuron. For each flattened image `x`, the model computes `a = sigmoid(w^T x + b)`, where `a` is interpreted as the probability of class 1.

The useful idea is that every later neural network layer is built from this same affine-plus-activation pattern, so keeping the math visible here makes the deeper notebooks easier to read.

Import NumPy. This stage uses NumPy directly so the matrix operations, gradients, and parameter updates stay visible instead of hiding inside a neural-network framework.


In [ ]:
import numpy as np

Load the saved image dataset from disk. The notebook expects `data/gopher_dataset.npz` to contain the train and test splits.


In [ ]:
loaded_data = np.load("data/gopher_dataset.npz")

Inspect the loaded archive. In Jupyter this displays the available arrays inside the `.npz` file.


In [ ]:
loaded_data 

Pull the train and test arrays out of the archive so later cells can work with normal NumPy variables.


In [ ]:
X_train = loaded_data["X_train"]
y_train = loaded_data["y_train"]
X_test = loaded_data["X_test"]
y_test = loaded_data["y_test"]

Check the raw training shapes before preprocessing. If there are `m` images, later losses and gradients average over `m`, so these dimensions determine the scale of `dW`, `db`, and accuracy metrics.


In [ ]:
X_train.shape
y_train.shape

Flatten each image tensor from `(height, width, channels)` into one vector `x` so it can multiply with `w`. Dividing pixels by 255 maps values into `[0, 1]`, which keeps feature magnitudes comparable and makes gradient-descent steps easier to tune.

Labels are reshaped into `(m, 1)` so the predicted probabilities and true labels have matching shapes in the loss calculation.


In [ ]:
X_flat = X_train.reshape(X_train.shape[0], -1) / 255.0
X_test = X_test.reshape(X_test.shape[0], -1) / 255.0
y_train = y_train.reshape(y_train.shape[0], 1)
y_test = y_test.reshape(y_test.shape[0], 1)
X_flat.shape

Initialize the logistic-regression parameters. The model starts with `w = 0` and `b = 0`; this is fine for single-neuron logistic regression because the binary cross-entropy objective is convex and there are no hidden units that need symmetry breaking.


In [ ]:
def initialize_parameters(X_flat):
    w = np.zeros((X_flat.shape[1], 1))
    b = 0
    return w, b

Create the initial parameters and inspect the weight shape. For `X` with shape `(m, n)`, `w` should have shape `(n, 1)` so `X @ w + b` returns one logit per image.


In [ ]:
w, b = initialize_parameters(X_flat)
w.shape

Define the forward pass. The affine score `z = Xw + b` measures evidence for class 1; the sigmoid `1 / (1 + exp(-z))` squashes that score into a probability between 0 and 1.

This is useful because probabilities can be compared directly with binary labels in cross-entropy.


In [ ]:
def forward_pass(X, w, b):
    z = np.dot(X, w) + b
    a = 1 / (1 + np.exp(-z))        
    return a

Define one gradient-descent step. Binary cross-entropy is `J = -mean(y log(a) + (1-y) log(1-a))`; with a sigmoid output, the derivative with respect to the logit simplifies to `dz = a - y`.

The gradients `dW = X.T @ dz / m` and `db = mean(dz)` point uphill in loss, so subtracting `learning_rate * gradient` moves the parameters downhill.


In [ ]:
def gradient_descent(X, Y, w, b, learning_rate = 0.01):
    y_pred = forward_pass(X, w, b)
    epsilon = 1e-7
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
    cost = -np.mean(Y * np.log(y_pred_clipped) + (1 - Y) * np.log(1 - y_pred_clipped))
    dz = y_pred - Y
    dW = (1 / X.shape[0]) * np.dot(X.T, dz)
    db = (1 / X.shape[0]) * np.sum(dz)
    w = w - learning_rate * dW
    b = b - learning_rate * db
    return w, b, cost



Train the logistic-regression model by repeating the same downhill update over many epochs. Watching the cost is useful because a steadily decreasing cost usually means the gradients, shapes, and learning rate are behaving sensibly.


In [ ]:
def train_model(X, Y, learning_rate = 0.01, epoch = 500):
    w, b = initialize_parameters(X)
    for i in range(epoch):
        w, b, cost = gradient_descent(X, Y, w, b)
        print(f"Cost for epoch {i} is {cost}")
    return w, b


Convert probabilities into class labels with a 0.5 threshold. In logit terms, 0.5 means `z = 0`: positive evidence predicts class 1, negative evidence predicts class 0.


In [ ]:
def predict(pred):
    return (pred > 0.5).astype(int)

Fit the model on the training set, then run the learned `w` and `b` on held-out test examples. This separates optimization performance from generalization performance.


In [ ]:
w, b = train_model(X_flat, y_train)
pred = forward_pass(X_test, w, b)

Measure train and test accuracy, then inspect one prediction visually. Accuracy summarizes the thresholded decisions, while the probability shows confidence before the hard 0/1 cutoff.


In [ ]:
import matplotlib.pyplot as plt

train_acc = (predict(forward_pass(X_flat, w, b)) == y_train).mean()
test_acc = (predict(forward_pass(X_test, w, b)) == y_test).mean()
print(f"train acc: {train_acc:.3f}   test acc: {test_acc:.3f}")

test_image = X_test[100].reshape(1, -1)
prediction_prob = forward_pass(test_image, w, b)
prediction = predict(prediction_prob)

img_display = (X_test[100].reshape(32, 32, 3) * 255).astype(np.uint8)

plt.figure(figsize=(6, 6))
plt.imshow(img_display)
plt.title(f"Pred: {'Gopher' if prediction[0][0] == 1 else 'Not Gopher'} ({prediction_prob[0][0]:.4f}) | True: {'Gopher' if y_test[0][0] == 1 else 'Not Gopher'}")
plt.axis('off')
plt.show()


Repeat the visualization on a different test image. A single accuracy number can hide systematic failures, so checking examples helps connect the math back to real images.


In [ ]:
import matplotlib.pyplot as plt

train_acc = (predict(forward_pass(X_flat, w, b)) == y_train).mean()
test_acc = (predict(forward_pass(X_test, w, b)) == y_test).mean()
print(f"train acc: {train_acc:.3f}   test acc: {test_acc:.3f}")

test_image = X_test[0].reshape(1, -1)
prediction_prob = forward_pass(test_image, w, b)
prediction = predict(prediction_prob)

img_display = (X_test[0].reshape(32, 32, 3) * 255).astype(np.uint8)

plt.figure(figsize=(6, 6))
plt.imshow(img_display)
plt.title(f"Pred: {'Gopher' if prediction[0][0] == 1 else 'Not Gopher'} ({prediction_prob[0][0]:.4f}) | True: {'Gopher' if y_test[0][0] == 1 else 'Not Gopher'}")
plt.axis('off')
plt.show()


Display a grid of test images with true labels. This is a qualitative check of the data distribution the model is trying to learn.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# show a grid of test images with true labels
fig, axes = plt.subplots(4, 6, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    img = (X_test[i].reshape(32, 32, 3) * 255).astype(np.uint8)
    ax.imshow(img)
    ax.set_title(f"y={int(y_test[i][0])}")
    ax.axis('off')
plt.tight_layout()
plt.show()


Reload and display raw examples from both classes. Seeing positives and negatives side by side helps explain why pixel features alone may be limited: the model must learn visual cues that distinguish similar small mammals.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("data/gopher_dataset.npz")
Xtr_raw, ytr_raw = d["X_train"], d["y_train"]
Xte_raw, yte_raw = d["X_test"], d["y_test"]
print("shapes:", Xtr_raw.shape, ytr_raw.shape, Xte_raw.shape, yte_raw.shape)

# show 6 gophers, 6 non-gophers
gophers = Xtr_raw[ytr_raw == 1][:6]
nongophers = Xtr_raw[ytr_raw == 0][:6]

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(gophers[i]); axes[0, i].axis("off"); axes[0, i].set_title("gopher")
    axes[1, i].imshow(nongophers[i]); axes[1, i].axis("off"); axes[1, i].set_title("not gopher")
plt.tight_layout(); plt.show()
